In [1]:
import pandas as pd

data = {
    "order_id": [1, 2, 2, 3, 4, 5],
    "customer": ["Amit", "Riya", "Riya", None, "Karan", "Neha"],
    "amount": ["500", "700", "700", "abc", "1200", None],
    "city": ["Jaipur", "Delhi", "Delhi", "Jaipur", "jaipur", "Mumbai"],
}

df = pd.DataFrame(data)
print(df)
df.info()

   order_id customer amount    city
0         1     Amit    500  Jaipur
1         2     Riya    700   Delhi
2         2     Riya    700   Delhi
3         3     None    abc  Jaipur
4         4    Karan   1200  jaipur
5         5     Neha   None  Mumbai
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   order_id  6 non-null      int64 
 1   customer  5 non-null      object
 2   amount    5 non-null      object
 3   city      6 non-null      object
dtypes: int64(1), object(3)
memory usage: 324.0+ bytes


In [2]:
df=df.drop_duplicates()
print(df)


   order_id customer amount    city
0         1     Amit    500  Jaipur
1         2     Riya    700   Delhi
3         3     None    abc  Jaipur
4         4    Karan   1200  jaipur
5         5     Neha   None  Mumbai


In [3]:
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df.info()
print(df)

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 0 to 5
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  5 non-null      int64  
 1   customer  4 non-null      object 
 2   amount    3 non-null      float64
 3   city      5 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 200.0+ bytes
   order_id customer  amount    city
0         1     Amit   500.0  Jaipur
1         2     Riya   700.0   Delhi
3         3     None     NaN  Jaipur
4         4    Karan  1200.0  jaipur
5         5     Neha     NaN  Mumbai


In [4]:
df["city"] = df["city"].str.strip().str.title()
print(df["city"].value_counts())

print(df.isna().sum())

city
Jaipur    3
Delhi     1
Mumbai    1
Name: count, dtype: int64
order_id    0
customer    1
amount      2
city        0
dtype: int64


In [5]:
df["customer"] = df["customer"].fillna("Unknown")

df["amount_filled"] = df["amount"].isna()
df["amount"] = df["amount"].fillna(df["amount"].median())

print(df)

   order_id customer  amount    city  amount_filled
0         1     Amit   500.0  Jaipur          False
1         2     Riya   700.0   Delhi          False
3         3  Unknown   700.0  Jaipur           True
4         4    Karan  1200.0  Jaipur          False
5         5     Neha   700.0  Mumbai           True


In [6]:
import sqlite3

conn = sqlite3.connect("orders.db")
df.to_sql("orders", conn, if_exists="replace", index=False)

query = """
SELECT city,
       COUNT(*) AS num_orders,
       SUM(amount) AS total_amount
FROM orders
GROUP BY city
ORDER BY total_amount DESC
"""
print(pd.read_sql(query, conn))

     city  num_orders  total_amount
0  Jaipur           3        2400.0
1  Mumbai           1         700.0
2   Delhi           1         700.0


In [7]:
customers = pd.DataFrame({
    "customer": ["Amit", "Riya", "Karan", "Neha"],
    "tier": ["Gold", "Silver", "Gold", "Silver"],
})
customers.to_sql("customers", conn, if_exists="replace", index=False)

# HAVING: only cities with total sales above 1000
q1 = """
SELECT city, COUNT(*) AS num_orders, SUM(amount) AS total
FROM orders
GROUP BY city
HAVING SUM(amount) > 1000
"""
print(pd.read_sql(q1, conn))

# JOIN: total sales per customer tier
q2 = """
SELECT c.tier, COUNT(*) AS num_orders, SUM(o.amount) AS total
FROM orders o
LEFT JOIN customers c ON o.customer = c.customer
GROUP BY c.tier
"""
print(pd.read_sql(q2, conn))

     city  num_orders   total
0  Jaipur           3  2400.0
     tier  num_orders   total
0    None           1   700.0
1    Gold           2  1700.0
2  Silver           2  1400.0


In [8]:
import pandas as pd

def clean_orders(df):
    df = df.drop_duplicates()
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["city"] = df["city"].str.strip().str.title()
    df["customer"] = df["customer"].fillna("Unknown")
    df["amount_filled"] = df["amount"].isna()
    df["amount"] = df["amount"].fillna(df["amount"].median())
    return df

raw = pd.DataFrame({
    "order_id": [1, 2, 2, 3, 4, 5],
    "customer": ["Amit", "Riya", "Riya", None, "Karan", "Neha"],
    "amount": ["500", "700", "700", "abc", "1200", None],
    "city": ["Jaipur", "Delhi", "Delhi", "Jaipur", "jaipur", "Mumbai"],
})

cleaned = clean_orders(raw)
print(cleaned)

   order_id customer  amount    city  amount_filled
0         1     Amit   500.0  Jaipur          False
1         2     Riya   700.0   Delhi          False
3         3  Unknown   700.0  Jaipur           True
4         4    Karan  1200.0  Jaipur          False
5         5     Neha   700.0  Mumbai           True


/tmp/ipykernel_5810/1964261608.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
/tmp/ipykernel_5810/1964261608.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["city"] = df["city"].str.strip().str.title()
/tmp/ipykernel_5810/1964261608.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas

In [9]:
assert cleaned["order_id"].is_unique, "Duplicate order_ids found"
assert cleaned["amount"].isna().sum() == 0, "Missing amounts remain"
assert set(cleaned["city"]) == {"Jaipur", "Delhi", "Mumbai"}, "City names not cleaned"
assert len(cleaned) == 5, "Wrong number of rows"
print("All tests passed")

All tests passed


In [10]:
print(len(cleaned))
print(cleaned["order_id"].tolist())
print(raw.duplicated().sum())

5
[1, 2, 3, 4, 5]
1


In [11]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/clean", methods=["POST"])
def clean_endpoint():
    orders = request.get_json()
    df = pd.DataFrame(orders)
    cleaned_df = clean_orders(df)
    return jsonify(cleaned_df.to_dict(orient="records"))

In [12]:
client = app.test_client()

messy_orders = [
    {"order_id": 1, "customer": "Amit", "amount": "500", "city": "Jaipur"},
    {"order_id": 2, "customer": "Riya", "amount": "700", "city": "Delhi"},
    {"order_id": 2, "customer": "Riya", "amount": "700", "city": "Delhi"},
    {"order_id": 3, "customer": None, "amount": "abc", "city": "jaipur"},
]

response = client.post("/clean", json=messy_orders)
print(response.status_code)
print(response.get_json())

200
[{'amount': 500.0, 'amount_filled': False, 'city': 'Jaipur', 'customer': 'Amit', 'order_id': 1}, {'amount': 700.0, 'amount_filled': False, 'city': 'Delhi', 'customer': 'Riya', 'order_id': 2}, {'amount': 600.0, 'amount_filled': True, 'city': 'Jaipur', 'customer': 'Unknown', 'order_id': 3}]


/tmp/ipykernel_5810/1964261608.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
/tmp/ipykernel_5810/1964261608.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["city"] = df["city"].str.strip().str.title()
/tmp/ipykernel_5810/1964261608.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas